# EID2 — Enhanced Infectious Diseases Database

This notebook demonstrates the **EID2** accessor, which provides access to the
[Enhanced Infectious Diseases Database](https://eid2.liverpool.ac.uk/) maintained by the
LUCINDA team at the University of Liverpool. EID2 mines scientific literature and
NCBI/PubMed metadata to map the relationships between pathogens, their hosts and
vectors, and their global distribution.

> **Citation (CC BY 4.0):** Wardeh M, Risley C, McIntyre MK, Setzkorn C, Baylis M.
> *Database of host-pathogen and related species interactions, and their global
> distribution.* Scientific Data 2, 150049 (2015). https://doi.org/10.1038/sdata.2015.49

The bulk datasets are distributed via FigShare and cached locally on first use.

In [ ]:
from epidatasets.sources.eid2 import EID2Accessor

eid2 = EID2Accessor()
print(eid2)
print(eid2.info())

## 1. Available datasets

EID2 distributes three bulk CSVs on FigShare: species-species interactions (2014),
species-location interactions (2014) and updated raw organism interactions (2022,
which additionally carry NCBI taxonomic identifiers).

In [ ]:
eid2.list_datasets()

## 2. Species-species interactions

The first call downloads `SpeciesInteractions_EID2.csv` (~4 MB) and caches it under
`~/.cache/epi_data/eid2/`. Each row is a carrier-cargo interaction backed by the
number of supporting NCBI sequences and PubMed publications (identifier lists included).

In [ ]:
interactions = eid2.get_interactions()
print(f"{len(interactions):,} interaction records")
interactions.head()

### 2.1 Filtering

All filters are case-insensitive and accept a single value or a list.

In [ ]:
# Virus interactions involving humans, backed by publications
human_viruses = eid2.get_interactions(
    carrier="homo sapiens",
    cargo_classification="Virus",
    has_publications=True,
)
print(f"{len(human_viruses):,} human-virus interactions")
human_viruses[["cargo", "sequences_count", "publications_count"]].head(10)

In [ ]:
# A specific pathogen across all carriers
eid2.get_interactions(cargo="zika virus")

### 2.2 Evidence identifiers

The `sequences` and `publications` columns hold `;`-separated NCBI GI / PubMed PMID
lists. Use `parse_evidence` to expand them.

In [ ]:
EID2Accessor.parse_evidence(human_viruses.iloc[0]["publications"])[:5]

## 3. Species-location records

The locations dataset links species to countries and first administrative divisions
(GeoNames free text, lower case).

In [ ]:
locations = eid2.get_locations()
print(f"{len(locations):,} location records covering {locations['country'].nunique()} countries")
locations.head()

In [ ]:
# Countries covered, most-recorded first
eid2.list_countries().head(10)

In [ ]:
# Virus records in Brazil, with the regions where they were observed
brazil_viruses = eid2.get_locations(country="brazil", classification="Virus")
print(f"{len(brazil_viruses):,} virus location records in Brazil")
brazil_viruses[["species", "region"]].head(10)

In [ ]:
# First administrative divisions covered
eid2.list_regions(country="brazil").head(10)

## 4. Species and classification overview

In [ ]:
# Most-connected species across the interaction network
species = eid2.list_species()
species.head(10)

In [ ]:
# Organism classifications (cargos are usually pathogens; carriers hosts/vectors)
eid2.list_classifications().head(10)

## 5. Updated 2022 interactions (with NCBI taxonomic IDs)

The 2022 snapshot re-publishes the interaction pairs with NCBI taxonomy
identifiers. The file is ~50 MB and is cached on first download.

In [ ]:
raw = eid2.get_raw_interactions_2022(cargo="zika virus")
raw

## 6. Quick analysis: top carriers of human viruses

In [ ]:
import matplotlib.pyplot as plt

top_carriers = (
    human_viruses.groupby("carrier")["publications_count"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

ax = top_carriers.sort_values().plot.barh(figsize=(8, 4), legend=False)
ax.set_xlabel("PubMed publications supporting the interactions")
ax.set_ylabel("")
ax.set_title("Top carriers of human viruses in EID2 (by publication evidence)")
plt.tight_layout()
plt.show()

## 7. Summary statistics

In [ ]:
eid2.get_summary_statistics()

## Notes and limitations

- EID2 is a **presence-only** database: the absence of a pathogen in a host cannot
  be inferred.
- Species names follow NCBI Taxonomy conventions of the snapshot's extraction date
  (e.g. `homo sapiens` rather than `human`); filters are exact-match
  (case-insensitive), so check `list_species()` for the exact spelling.
- The 2014 FigShare snapshots are static; the 2022 raw file reflects a later
  extraction. Automated scraping of the EID2 **web interface** is prohibited —
  this accessor only uses the sanctioned bulk downloads.
- Data are CC BY 4.0 — cite Wardeh et al. (2015) when using them.